In [ ]:
!pip install -r requirements.txt
from warpdrive import WarpDrive
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objs as go
from geopy.geocoders import Nominatim
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import folium
import pickle
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression

wd = WarpDrive()

df = wd.get_args("df")
col = wd.get_args("col")
strr = wd.get_args("strr")
int = wd.get_args("int")
float = wd.get_args("float")
bool = wd.get_args("bool")
cf = wd.get_args("cf")

# --- Model Training and Saving ---
X = df[['Loan_Amount', 'Home_Owner']]  # Features
y = df['Gender']  # Target

# Step 3: Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: Train a Logistic Regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Step 5: Evaluate the model (optional)
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
exog_columns = ["Loan_Amount", "Home_Owner"]
wd.create_model(
    model=model,
    library="sklearn",
    model_technique="LogisticRegressionClassifier",
    input_variables=exog_columns,
    target_column="Gender",
    train_table="df",
    lags=0,
    exog_columns=exog_columns
)
with open('logistic_model.pkl', 'wb') as file:
    pickle.dump(model, file)

clf = CatBoostClassifier(random_state=0)
clf.fit(df[exog_columns].values, df['Gender'].values)
col = exog_columns
wd.create_model(
    model=clf,
    library="catboost",
    model_technique="CatboostClassifier",
    input_variables=col,
    target_column="Gender",
    train_table="df",
    lags=0,
    exog_columns=exog_columns
)
with open('cbc_model.pkl', 'wb') as file:
    pickle.dump(clf, file)

# --- DataFrame creation from series ---
s1 = pd.Series([1, 3, 4, 5, 6, 2])
s2 = pd.Series([1.1, 3.5, 4.7, 5.8, 2.9, 9.3])
s3 = pd.Series(['a', 'b', 'c', 'd', 'e','f'])
Data = {'first': s1, 'second': s2, 'third': s3}
df = pd.DataFrame(Data)
for i in range(0, 10):
    wd.create_df(df)

# --- Simple Line Plot ---
x = [1, 2, 3, 4, 5]
y = [2, 3, 5, 7, 11]
plt.plot(x, y, marker='o', linestyle='-')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.title('Simple Line Plot')
plt.grid(True)
wd.save_image(plt)
plt.show()

# --- Bar Chart ---
categories = ['A', 'B', 'C', 'D']
values = [10, 20, 15, 25]
trace = go.Bar(x=categories, y=values)
layout = go.Layout(title='Bar Chart Example', xaxis=dict(title='Categories'), yaxis=dict(title='Values'))
fig = go.Figure(data=[trace], layout=layout)
wd.save_graph(fig)

# --- Financial DataFrame and Plot ---
data = {
    'Year': [2024, 2024, 2025, 2025, 2026, 2026],
    'Scenario': ['Baseline', 'Stress', 'Baseline', 'Stress', 'Baseline', 'Stress'],
    'CET1': [10.50, 8.2, 11.70, 9.9, 12.70, 11.3],
    'Tier 1': [13.80, 11.3, 14.70, 12.6, 15.30, 13.7],
    'Total Capital': [15.00, 12.6, 15.90, 13.9, 16.50, 15]
}
df = pd.DataFrame(data)
wd.save_table(df, "zits-soly")

fig, ax = plt.subplots(figsize=(10, 6))
for column in ['CET1', 'Tier 1', 'Total Capital']:
    for scenario in df['Scenario'].unique():
        df_subset = df[df['Scenario'] == scenario]
        ax.plot(df_subset['Year'], df_subset[column], marker='o', label=f'{scenario} {column}')
ax.set_xlabel('Year')
ax.set_ylabel('Percentage')
ax.set_title('Financial Metrics over Years by Scenario')
ax.legend(loc='upper left')
plt.tight_layout()
wd.save_image(plt, "zits-soly")

# --- GeoLocation and Mapping ---
geolocator = Nominatim(user_agent="location_details")
location = geolocator.geocode("Kanpur")
print((location.latitude, location.longitude))
map = folium.Map(location=[location.latitude, location.longitude], zoom_start=15)

# --- Plotly Grouped Bar Chart ---
df = pd.DataFrame(data)
subgroups = df['Scenario'].unique()
traces = []
for subgroup in subgroups:
    subgroup_df = df[df['Scenario'] == subgroup]
    for i, column in enumerate(df.columns[2:]):
        traces.append(go.Bar(
            x=subgroup_df['Year'] + i * 0.2,
            y=subgroup_df[column],
            name=f'{column} ({subgroup})',
            offsetgroup=i,
            width=0.2
        ))
layout = go.Layout(
    title='Capital Ratio Over Time by Scenario',
    xaxis=dict(title='Year'),
    yaxis=dict(title='Capital Ratio'),
    barmode='group'
)
fig = go.Figure(data=traces, layout=layout)
wd.save_graph(fig, "shiv-soly")
wd.add_output_artifact("INTEGER_OP", 5)
wd.add_output_artifact("FLOAT_OP", 11.11)
wd.add_output_artifact("BOOLEAN_OP", False)
wd.add_output_artifact("STRING_OP", "String_Output")
wd.save_console_file("cf")

